# Final Analysis
## Indonesia & Russia — RCP 4.5 Bias-Corrected Ensemble
Loads per-model BC PRCPTOT NetCDFs from `bias_correction_Indonesia.ipynb` and `bias_correction_Russia.ipynb`, rebuilds ensembles, and produces all publication figures and summary tables.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.ndimage import zoom as _ndz
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.titlesize': 13,
})
%matplotlib inline
print('Libraries imported.')


## Step 1: Configuration & paths

In [ ]:
PROJECT_DIR   = r'[PROJECT_DIR]'
BC_INDO_DIR   = os.path.join(PROJECT_DIR, 'data', 'bias_corrected', 'Indonesia')
BC_RUS_DIR    = os.path.join(PROJECT_DIR, 'data', 'bias_corrected', 'Russia')
OUT_DIR       = os.path.join(PROJECT_DIR, 'data', 'final_analysis')
os.makedirs(OUT_DIR, exist_ok=True)

MODELS  = ['HadGEM2-AO', 'MPI-ESM-MR', 'IPSL-CM5A-LR']
COLORS  = {'HadGEM2-AO': '#E63946', 'MPI-ESM-MR': '#2196F3', 'IPSL-CM5A-LR': '#4CAF50'}
SEASONS = {'DJF': [12,1,2], 'MAM': [3,4,5], 'JJA': [6,7,8], 'SON': [9,10,11]}

INDO_EXTENT    = [94, 141, -11, 6]
INDO_PROJ      = ccrs.PlateCarree()
RUSSIA_EXTENT  = [19, 192, 40, 83]
RUSSIA_PROJ    = ccrs.PlateCarree(central_longitude=105)

INDO_BASELINE  = '1981\u20132005'
RUS_BASELINE   = '1976\u20132005'
FUTURE_LABEL   = '2041\u20132070'

print(f'Output directory : {OUT_DIR}')
print(f'Models           : {MODELS}')


## Step 2: Visualisation helpers

In [ ]:
_OCEAN = cfeature.NaturalEarthFeature(
    'physical', 'ocean', '10m', facecolor='white', edgecolor='none')
_LAKES = cfeature.NaturalEarthFeature(
    'physical', 'lakes', '10m', facecolor='white', edgecolor='none')

def _add_ocean_mask(ax):
    ax.add_feature(_OCEAN, zorder=2)
    ax.add_feature(_LAKES, zorder=2)

def _upsample(data, factor, order):
    filled = np.where(np.isnan(data), 0.0, data)
    return _ndz(filled, factor, order=order)

def _smooth_and_pad(da, out_step=0.25, pad_deg=2):
    """
    Resample DataArray to `out_step` degree grid, extending `pad_deg` beyond the data range.
    Uses np.pad(mode='edge') so border cells carry the nearest real value.
    This guarantees no NaN at any output point.
    """
    da = da.sortby('lat')
    lats = da.lat.values.astype(float)
    lons = da.lon.values.astype(float)
    data = np.where(np.isnan(da.values.astype(float)),
                    np.nanmean(da.values), da.values.astype(float))

    lat_step = float(np.diff(lats).mean())
    lon_step = float(np.diff(lons).mean())
    n_pad_lat = max(1, int(np.ceil(pad_deg / lat_step)))
    n_pad_lon = max(1, int(np.ceil(pad_deg / lon_step)))

    data_pad = np.pad(data, ((n_pad_lat, n_pad_lat), (n_pad_lon, n_pad_lon)),
                      mode='edge')
    lats_pad = np.concatenate([
        lats[0]  - np.arange(n_pad_lat, 0, -1) * lat_step,
        lats,
        lats[-1] + np.arange(1, n_pad_lat + 1) * lat_step,
    ])
    lons_pad = np.concatenate([
        lons[0]  - np.arange(n_pad_lon, 0, -1) * lon_step,
        lons,
        lons[-1] + np.arange(1, n_pad_lon + 1) * lon_step,
    ])
    da_pad   = xr.DataArray(data_pad, dims=['lat','lon'],
                             coords={'lat': lats_pad, 'lon': lons_pad})
    fine_lats = np.arange(lats_pad[0],  lats_pad[-1]  + out_step, out_step)
    fine_lons = np.arange(lons_pad[0],  lons_pad[-1]  + out_step, out_step)
    return da_pad.interp(lat=fine_lats, lon=fine_lons, method='linear')


def _add_style(ax, im, unit, cbar_pad, shrink, ticks, label_size, extent):
    _add_ocean_mask(ax)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6,
                   alpha=0.8, zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linestyle='--',
                      linewidth=0.4, alpha=0.5, color='gray', zorder=4)
    gl.top_labels = False; gl.right_labels = False
    gl.xlabel_style = {'size': label_size - 1}
    gl.ylabel_style = {'size': label_size - 1}
    cbar = plt.colorbar(im, ax=ax, pad=cbar_pad, shrink=shrink, ticks=ticks)
    cbar.ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{int(x)}'))
    cbar.ax.tick_params(labelsize=label_size - 1)
    return cbar


def plot_div(ax, da, vmin, vmax, extent, cmap='RdBu_r', center=0,
             cbar_pad=0.10, shrink=0.65, unit='mm/yr',
             tick_step=None, label_size=9):
    norm  = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    da_sm = _smooth_and_pad(da)
    lf, af = da_sm.lon.values, da_sm.lat.values
    d_up  = _upsample(da_sm.values, factor=4, order=1)
    lu    = np.linspace(lf[0], lf[-1], d_up.shape[1])
    au    = np.linspace(af[0], af[-1], d_up.shape[0])
    im    = ax.pcolormesh(lu, au, d_up, cmap=cmap, norm=norm,
                          transform=ccrs.PlateCarree(),
                          shading='gouraud', zorder=1)
    if tick_step is None:
        span = vmax - vmin
        tick_step = next((s for s in [200,100,50,25,10,5] if span/s <= 12), 50)
    t_min = int(np.ceil(vmin  / tick_step)) * tick_step
    t_max = int(np.floor(vmax / tick_step)) * tick_step
    ticks = list(range(t_min, t_max + 1, tick_step))
    cbar  = _add_style(ax, im, unit, cbar_pad, shrink, ticks, label_size, extent)
    cbar.set_label(unit, size=label_size)
    return im


def plot_seq(ax, da, vmin, vmax, extent, cmap='Blues',
             cbar_pad=0.10, shrink=0.65, unit='mm/yr',
             tick_step=None, label_size=9):
    norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)
    da_sm = _smooth_and_pad(da)
    lf, af = da_sm.lon.values, da_sm.lat.values
    d_up  = _upsample(da_sm.values, factor=4, order=1)
    lu    = np.linspace(lf[0], lf[-1], d_up.shape[1])
    au    = np.linspace(af[0], af[-1], d_up.shape[0])
    im    = ax.pcolormesh(lu, au, d_up, cmap=cmap, norm=norm,
                          transform=ccrs.PlateCarree(),
                          shading='gouraud', zorder=1)
    if tick_step is None:
        span = vmax - vmin
        tick_step = next((s for s in [100,50,25,10,5] if span/s <= 10), 25)
    t_min = int(np.ceil(vmin  / tick_step)) * tick_step
    t_max = int(np.floor(vmax / tick_step)) * tick_step
    ticks = list(range(t_min, t_max + 1, tick_step))
    cbar  = _add_style(ax, im, unit, cbar_pad, shrink, ticks, label_size, extent)
    cbar.set_label(unit, size=label_size)
    return im


def plot_stress(ax, da, extent, cbar_pad=0.10, shrink=0.65, label_size=9):
    cmap_ws = mcolors.ListedColormap(['#4CAF50','#FFC107','#F44336'])
    norm_ws = mcolors.BoundaryNorm([0.5,1.5,2.5,3.5], ncolors=3)
    da_sm   = _smooth_and_pad(da)
    lf, af  = da_sm.lon.values, da_sm.lat.values
    d_up    = _upsample(np.round(da_sm.values), factor=4, order=0)
    lu      = np.linspace(lf[0], lf[-1], d_up.shape[1])
    au      = np.linspace(af[0], af[-1], d_up.shape[0])
    im      = ax.pcolormesh(lu, au, d_up, cmap=cmap_ws, norm=norm_ws,
                            transform=ccrs.PlateCarree(),
                            shading='nearest', zorder=1)
    _add_ocean_mask(ax)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6,
                   alpha=0.8, zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linestyle='--',
                      linewidth=0.4, alpha=0.5, color='gray', zorder=4)
    gl.top_labels = False; gl.right_labels = False
    gl.xlabel_style = {'size': label_size - 1}
    gl.ylabel_style = {'size': label_size - 1}
    cbar = plt.colorbar(im, ax=ax, pad=cbar_pad, shrink=shrink, ticks=[1,2,3])
    cbar.ax.set_yticklabels(
        ['Low\n(WSI \u2265 \u221210%)',
         'Moderate\n(\u221220% \u2264 WSI < \u221210%)',
         'High\n(WSI < \u221220%)'],
        fontsize=label_size - 1)
    return im

print('Visualisation helpers ready.')


## Step 3: Load bias-corrected PRCPTOT

In [ ]:
def load_bc_model(bc_dir, region):
    data = {}
    for mn in MODELS:
        safe = mn.replace('-', '_')
        f_base = os.path.join(bc_dir, f'PRCPTOT_bc_baseline_{safe}_{region}.nc')
        f_fut  = os.path.join(bc_dir, f'PRCPTOT_bc_future_{safe}_{region}.nc')
        if not os.path.exists(f_base):
            print(f'  \u2717 Missing: {f_base}'); continue
        if not os.path.exists(f_fut):
            print(f'  \u2717 Missing: {f_fut}');  continue
        ds_b = xr.open_dataset(f_base);  var_b = list(ds_b.data_vars)[0]
        ds_f = xr.open_dataset(f_fut);   var_f = list(ds_f.data_vars)[0]
        data[mn] = {
            'baseline_ts':   ds_b[var_b],
            'future_ts':     ds_f[var_f],
            'baseline_mean': ds_b[var_b].mean(dim='time'),
            'future_mean':   ds_f[var_f].mean(dim='time'),
        }
        data[mn]['anomaly'] = data[mn]['future_mean'] - data[mn]['baseline_mean']
        print(f'  \u2713 {mn:15s}  '
              f'base={float(data[mn]["baseline_mean"].mean()):.1f}  '
              f'fut={float(data[mn]["future_mean"].mean()):.1f}  '
              f'anom={float(data[mn]["anomaly"].mean()):+.1f} mm/yr')
    return data

print('=' * 60)
print('LOADING BIAS-CORRECTED PRCPTOT \u2014 INDONESIA')
print('=' * 60)
indo_bc = load_bc_model(BC_INDO_DIR, 'Indonesia')

print()
print('=' * 60)
print('LOADING BIAS-CORRECTED PRCPTOT \u2014 RUSSIA')
print('=' * 60)
rus_bc = load_bc_model(BC_RUS_DIR, 'Russia')

print(f'\nLoaded {len(indo_bc)}/3 Indonesia models  |  {len(rus_bc)}/3 Russia models')


## Step 4: Build multi-model ensembles & CWSI

In [ ]:
def build_ensemble(bc_data):
    """
    Regrid all models to the first model's grid, then compute:
    ensemble mean/std, WSI, CWSI stress (NaN-safe), % model agreement.
    """
    ref_mn  = list(bc_data.keys())[0]
    ref_lat = bc_data[ref_mn]['baseline_mean'].lat
    ref_lon = bc_data[ref_mn]['baseline_mean'].lon

    base_list, fut_list, anom_list = [], [], []
    for mn, d in bc_data.items():
        rg = lambda da: da.interp(lat=ref_lat, lon=ref_lon, method='nearest')
        base_list.append(rg(d['baseline_mean']))
        fut_list.append( rg(d['future_mean']))
        anom_list.append(rg(d['anomaly']))

    base_stack = xr.concat(base_list, dim='model')
    fut_stack  = xr.concat(fut_list,  dim='model')
    anom_stack = xr.concat(anom_list, dim='model')

    ens_base = base_stack.mean(dim='model')
    ens_fut  = fut_stack.mean( dim='model')
    ens_anom = anom_stack.mean(dim='model')
    ens_std  = anom_stack.std( dim='model')

    # WSI, NaN where baseline is zero/missing
    ens_wsi  = xr.where(ens_base > 0,
                        (ens_fut - ens_base) / ens_base * 100,
                        np.nan)

    # CWSI stress.
    # Initialise with NaN so ocean pixels stay NaN
    stress = xr.full_like(ens_wsi, fill_value=np.nan)
    stress = xr.where(ens_wsi >= -10,                      1.0, stress)  # Low
    stress = xr.where((ens_wsi < -10) & (ens_wsi >= -20), 2.0, stress)  # Moderate
    stress = xr.where(ens_wsi < -20,                       3.0, stress)  # High

    # % models agreeing on wetter direction
    pct_inc = (anom_stack > 0).sum(dim='model') / len(bc_data) * 100

    return {
        'baseline':     ens_base,
        'future':       ens_fut,
        'anomaly':      ens_anom,
        'std':          ens_std,
        'wsi':          ens_wsi,
        'stress':       stress,
        'pct_increase': pct_inc,
        'anom_stack':   anom_stack,
    }


def _stress_pcts(stress):
    v = stress.values.ravel()
    v = v[~np.isnan(v)]
    n = len(v)
    if n == 0: return 0.0, 0.0, 0.0
    return (np.sum(v==1)/n*100, np.sum(v==2)/n*100, np.sum(v==3)/n*100)


ens_indo = build_ensemble(indo_bc)
ens_rus  = build_ensemble(rus_bc)

for region, ens in [('Indonesia', ens_indo), ('Russia', ens_rus)]:
    lo, mo, hi = _stress_pcts(ens['stress'])
    print(f'\n{region}:')
    print(f'  Baseline  : {float(ens["baseline"].mean()):.1f} mm/yr')
    print(f'  Future    : {float(ens["future"].mean()):.1f} mm/yr')
    print(f'  Anomaly   : {float(ens["anomaly"].mean()):+.1f} mm/yr')
    print(f'  Std Dev   : {float(ens["std"].mean()):.1f} mm/yr')
    print(f'  Mean WSI  : {float(ens["wsi"].mean()):+.1f} %')
    print(f'  CWSI      : Low {lo:.1f}%  Moderate {mo:.1f}%  High {hi:.1f}%')
    print(f'  Total     : {lo+mo+hi:.1f}%  (should = 100%)')


## Step 5: Seasonal decomposition
Uses monthly BC NetCDFs if present; falls back to annual anomaly proxy with clear labelling if not.

In [ ]:
def _seasonal_from_monthly(monthly_nc, season_months):
    ds  = xr.open_dataset(monthly_nc)
    var = list(ds.data_vars)[0]
    da  = ds[var]
    sea = da.isel(time=da.time.dt.month.isin(season_months))
    base = sea.isel(time=sea.time.dt.year.isin(range(1976, 2006))).mean('time')
    fut  = sea.isel(time=sea.time.dt.year.isin(range(2041, 2071))).mean('time')
    return base, fut, fut - base


def build_seasonal(bc_data, bc_dir, region):
    seasonal = {}
    for season, months in SEASONS.items():
        anom_list = []
        has_monthly = True
        for mn, d in bc_data.items():
            safe = mn.replace('-', '_')
            candidates = [
                os.path.join(bc_dir, f'pr_bc_future_{safe}_{region}_monthly.nc'),
                os.path.join(bc_dir, f'BC_monthly_{safe}_{region}.nc'),
            ]
            nc = next((p for p in candidates if os.path.exists(p)), None)
            if nc:
                _, _, anom = _seasonal_from_monthly(nc, months)
                anom_list.append(anom)
            else:
                has_monthly = False
                break
        if has_monthly and anom_list:
            ref = bc_data[list(bc_data.keys())[0]]['anomaly']
            ens_anom = xr.concat(
                [a.interp(lat=ref.lat, lon=ref.lon, method='nearest')
                 for a in anom_list],
                dim='model').mean('model')
            seasonal[season] = {'anomaly': ens_anom, 'proxy': False}
        else:
            # Fallback: annual ensemble anomaly as uniform proxy
            ref_mn  = list(bc_data.keys())[0]
            ref_lat = bc_data[ref_mn]['anomaly'].lat
            ref_lon = bc_data[ref_mn]['anomaly'].lon
            annual_anom = xr.concat(
                [d['anomaly'].interp(lat=ref_lat, lon=ref_lon, method='nearest')
                 for d in bc_data.values()],
                dim='model').mean('model')
            seasonal[season] = {'anomaly': annual_anom, 'proxy': True}
    return seasonal


seasonal_indo = build_seasonal(indo_bc, BC_INDO_DIR, 'Indonesia')
seasonal_rus  = build_seasonal(rus_bc,  BC_RUS_DIR,  'Russia')

mode = lambda s: 'monthly BC files' if not list(s.values())[0]['proxy'] else 'annual proxy'
print(f'Seasonal mode  Indonesia : {mode(seasonal_indo)}')
print(f'Seasonal mode  Russia    : {mode(seasonal_rus)}')

## Step 6: Figure 1: 4-panel Baseline / Future / Anomaly / Uncertainty

In [ ]:
# Figure 1: 4-panel: Baseline / Future / Anomaly / Uncertainty

def figure_4panel(ens, proj, extent, region, base_lbl, fut_lbl,
                  vmax_abs, tick_abs, tick_anom, tick_std,
                  vmax_std, outpath, shrink=0.65):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10),
                             subplot_kw={'projection': proj})
    fig.suptitle(
        f'{region} \u2014 Bias-Corrected PRCPTOT \u00b7 RCP 4.5\n'
        'EQM \u00b7 3-GCM Ensemble \u00b7 HadGEM2-AO \u00b7 MPI-ESM-MR \u00b7 IPSL-CM5A-LR',
        fontsize=13, fontweight='bold', y=1.01)

    axA, axB = axes[0];  axC, axD = axes[1]

    plot_seq(axA, ens['baseline'], 0, vmax_abs, extent,
             cmap='Blues', tick_step=tick_abs, shrink=shrink)
    axA.set_title(f'A  Corrected Baseline PRCPTOT\n({base_lbl})', fontweight='bold')

    plot_seq(axB, ens['future'], 0, vmax_abs, extent,
             cmap='Blues', tick_step=tick_abs, shrink=shrink)
    axB.set_title(f'B  Corrected Future PRCPTOT\n({fut_lbl})', fontweight='bold')

    # Round anomaly limits to nearest tick interval
    anom_abs = max(abs(float(ens['anomaly'].min())),
                   abs(float(ens['anomaly'].max())))
    anom_lim = max(int(np.ceil(anom_abs * 1.1 / tick_anom)) * tick_anom, tick_anom)
    plot_div(axC, ens['anomaly'], -anom_lim, anom_lim, extent,
             tick_step=tick_anom, shrink=shrink)
    axC.set_title(
        f'C  Anomaly (Future \u2212 Baseline)\n'
        f'Ensemble mean: {float(ens["anomaly"].mean()):+.1f} mm/yr',
        fontweight='bold')

    plot_seq(axD, ens['std'], 0, vmax_std, extent,
             cmap='YlOrRd', tick_step=tick_std, shrink=shrink)
    axD.set_title('D  Model Uncertainty\n(Ensemble Std Dev)', fontweight='bold')

    plt.tight_layout(pad=2.5, w_pad=3.0, h_pad=3.5)
    plt.savefig(outpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show();  print(f'Saved: {outpath}')


figure_4panel(ens_indo, INDO_PROJ, INDO_EXTENT, 'Indonesia',
              INDO_BASELINE, FUTURE_LABEL,
              vmax_abs=4000, tick_abs=500, tick_anom=100,
              tick_std=50,   vmax_std=400,
              outpath=os.path.join(OUT_DIR, 'Fig1_Indonesia_4panel.png'))

figure_4panel(ens_rus, RUSSIA_PROJ, RUSSIA_EXTENT, 'Russia',
              RUS_BASELINE, FUTURE_LABEL,
              vmax_abs=1200, tick_abs=200, tick_anom=40,
              tick_std=25,   vmax_std=150,
              outpath=os.path.join(OUT_DIR, 'Fig1_Russia_4panel.png'), shrink=0.42)


## Step 7: Figure 2: WSI continuous + CWSI categories

In [ ]:
# Figure 2: WSI continuous + CWSI categories

def figure_wsi(ens, proj, extent, region, outpath, shrink=0.65):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5),
                                    subplot_kw={'projection': proj})
    fig.suptitle(
        f'{region} \u2014 Climate Water Stress Index (CWSI) \u00b7 RCP 4.5 \u00b7 {FUTURE_LABEL}',
        fontsize=12, fontweight='bold')

    wsi_abs = max(abs(float(ens['wsi'].min())), abs(float(ens['wsi'].max())))
    wsi_lim = max(int(np.ceil(wsi_abs * 1.1 / 10)) * 10, 10)
    plot_div(ax1, ens['wsi'], -wsi_lim, wsi_lim, extent,
             tick_step=10, unit='WSI (%)', shrink=shrink)
    ax1.set_title(
        f'WSI \u2014 Continuous (%)\nMean: {float(ens["wsi"].mean()):+.1f}%',
        fontweight='bold')

    plot_stress(ax2, ens['stress'], extent, shrink=shrink)
    lo, mo, hi = _stress_pcts(ens['stress'])
    ax2.set_title(
        f'CWSI Categories\nLow {lo:.0f}%  \u00b7  Mod {mo:.0f}%  \u00b7  High {hi:.0f}%',
        fontweight='bold')

    plt.tight_layout(pad=2.0, w_pad=3.0)
    plt.savefig(outpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show();  print(f'Saved: {outpath}')


figure_wsi(ens_indo, INDO_PROJ, INDO_EXTENT, 'Indonesia',
           os.path.join(OUT_DIR, 'Fig2_Indonesia_WSI.png'))
figure_wsi(ens_rus,  RUSSIA_PROJ, RUSSIA_EXTENT, 'Russia',
           os.path.join(OUT_DIR, 'Fig2_Russia_WSI.png'), shrink=0.42)


## Step 8: Figure 3: Individual GCM anomalies vs ensemble

In [ ]:
# Figure 3: Individual GCM anomalies vs ensemble mean

def figure_model_comparison(bc_data, ens, proj, extent, region,
                              vmin, vmax, tick_step, outpath,
                              shrink=0.50, figsize=(22, 5)):
    fig, axes = plt.subplots(1, 4, figsize=figsize,
                              subplot_kw={'projection': proj})
    fig.suptitle(
        f'{region} \u2014 Individual GCM Anomalies vs Ensemble Mean (BC) \u00b7 RCP 4.5',
        fontsize=12, fontweight='bold', y=1.02)

    ref_mn  = list(bc_data.keys())[0]
    ref_lat = bc_data[ref_mn]['anomaly'].lat
    ref_lon = bc_data[ref_mn]['anomaly'].lon

    panels = list(bc_data.keys()) + ['_ensemble']
    for ax, panel in zip(axes, panels):
        if panel == '_ensemble':
            da    = ens['anomaly']
            title = f'Ensemble Mean (BC)\n{float(da.mean()):+.1f} mm/yr'
        else:
            da    = bc_data[panel]['anomaly'].interp(lat=ref_lat, lon=ref_lon,
                                                      method='nearest')
            title = f'{panel}\n{float(da.mean()):+.1f} mm/yr'
        plot_div(ax, da, vmin, vmax, extent,
                 tick_step=tick_step, shrink=shrink, label_size=8)
        ax.set_title(title, fontsize=9, fontweight='bold')

    plt.tight_layout(pad=1.5, w_pad=2.0)
    plt.savefig(outpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show();  print(f'Saved: {outpath}')


figure_model_comparison(
    indo_bc, ens_indo, INDO_PROJ, INDO_EXTENT, 'Indonesia',
    -400, 400, 100, os.path.join(OUT_DIR, 'Fig3_Indonesia_ModelComparison.png'),
    shrink=0.50, figsize=(22, 5))

figure_model_comparison(
    rus_bc, ens_rus, RUSSIA_PROJ, RUSSIA_EXTENT, 'Russia',
    -160, 160, 40,  os.path.join(OUT_DIR, 'Fig3_Russia_ModelComparison.png'),
    shrink=0.30, figsize=(28, 5))


## Step 9: Figure 4: Seasonal anomaly panels

In [ ]:
# Figure 4: Seasonal anomaly panels

def figure_seasonal(seasonal, ens_annual_anom, proj, extent, region,
                     vmin, vmax, tick_step, outpath, shrink=0.65):
    fig, axes = plt.subplots(1, 4, figsize=(24, 5),
                             subplot_kw={'projection': proj})
    labels = {'DJF':'Dec\u2013Feb','MAM':'Mar\u2013May',
              'JJA':'Jun\u2013Aug','SON':'Sep\u2013Nov'}
    proxy  = list(seasonal.values())[0]['proxy']
    note   = '\n(annual proxy)' if proxy else ''

    for ax, (season, lbl) in zip(axes, labels.items()):
        anom = seasonal[season]['anomaly'] if not proxy else ens_annual_anom
        plot_div(ax, anom, vmin, vmax, extent,
                 tick_step=tick_step, unit='mm/season',
                 shrink=shrink, label_size=8)
        ax.set_title(
            f'{season}  ({lbl}){note}\nMean: {float(anom.mean()):+.1f} mm',
            fontsize=9, fontweight='bold')

    fig.suptitle(
        f'{region} \u2014 Seasonal PRCPTOT Anomaly \u00b7 RCP 4.5 \u00b7 '
        f'{FUTURE_LABEL} vs baseline\nBias-Corrected Ensemble Mean',
        fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout(pad=2.0, w_pad=2.5)
    plt.savefig(outpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show();  print(f'Saved: {outpath}')


figure_seasonal(
    seasonal_indo, ens_indo['anomaly'],
    INDO_PROJ, INDO_EXTENT, 'Indonesia',
    -200, 200, 50, os.path.join(OUT_DIR, 'Fig4_Indonesia_Seasonal.png'))

figure_seasonal(
    seasonal_rus, ens_rus['anomaly'],
    RUSSIA_PROJ, RUSSIA_EXTENT, 'Russia',
    -80, 80, 20, os.path.join(OUT_DIR, 'Fig4_Russia_Seasonal.png'), shrink=0.42)


## Step 10:Figure 5: Annual PRCPTOT time series with ensemble spread

In [ ]:
# Figure 5: Annual PRCPTOT time series with ensemble spread

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

for ax, bc_data, ens, region, base_period in [
    (ax1, indo_bc, ens_indo, 'Indonesia', INDO_BASELINE),
    (ax2, rus_bc,  ens_rus,  'Russia',    RUS_BASELINE),
]:
    fut_arrays = []
    ref_yrs    = None

    for mn, d in bc_data.items():
        clr = COLORS[mn]
        # Ensure annual resolution
        base_da = d['baseline_ts']
        fut_da  = d['future_ts']
        if len(base_da.time) > 50: base_da = base_da.resample(time='YE').sum()
        if len(fut_da.time)  > 50: fut_da  = fut_da.resample( time='YE').sum()

        base_ts = base_da.mean(dim=['lat','lon'])
        fut_ts  = fut_da.mean( dim=['lat','lon'])

        ax.plot(base_ts.time.dt.year.values, base_ts.values,
                color=clr, lw=1.2, ls='--', alpha=0.6,
                label=f'{mn} baseline')
        ax.plot(fut_ts.time.dt.year.values, fut_ts.values,
                color=clr, lw=1.8, label=f'{mn} future')

        if ref_yrs is None:
            ref_yrs = fut_ts.time.dt.year.values
        yrs = fut_ts.time.dt.year.values
        arr = fut_ts.values if len(yrs)==len(ref_yrs) else np.interp(ref_yrs, yrs, fut_ts.values)
        fut_arrays.append(arr)

    fut_arr = np.array(fut_arrays)
    ax.fill_between(ref_yrs, fut_arr.min(0), fut_arr.max(0),
                    alpha=0.12, color='gray', label='Ensemble spread')
    ax.plot(ref_yrs, fut_arr.mean(0),
            color='black', lw=2.5, label='Ensemble mean future')
    ax.axvspan(ref_yrs[0], ref_yrs[-1], alpha=0.04, color='orange',
               label='Future window')
    ax.set_xlabel('Year'); ax.set_ylabel('PRCPTOT (mm/yr)')
    ax.set_title(f'{region} \u2014 Annual PRCPTOT Bias-Corrected GCMs\n'
                 f'Baseline {base_period} \u00b7 Future {FUTURE_LABEL}',
                 fontweight='bold')
    ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

plt.tight_layout(pad=2.5)
out = os.path.join(OUT_DIR, 'Fig5_TimeSeries_BothRegions.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show();  print(f'Saved: {out}')


## Step 11: Figure 6: Anomaly & model agreement maps

In [ ]:
# Figure 6: Anomaly & model agreement side-by-side per region

for ens, proj, extent, region, vmin, vmax, tick, shrink in [
    (ens_indo, INDO_PROJ,   INDO_EXTENT,   'Indonesia', -400, 400, 100, 0.65),
    (ens_rus,  RUSSIA_PROJ, RUSSIA_EXTENT, 'Russia',    -160, 160,  40, 0.42),
]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5),
                                    subplot_kw={'projection': proj})
    plot_div(ax1, ens['anomaly'], vmin, vmax, extent,
             tick_step=tick, shrink=shrink)
    ax1.set_title(f'{region} \u2014 Ensemble Mean Anomaly (BC)',
                  fontweight='bold')

    plot_div(ax2, ens['pct_increase'], 0, 100, extent,
             center=50, tick_step=25, unit='% Models', shrink=shrink)
    ax2.set_title(
        f'{region} \u2014 % Models Showing Wetter Signal\n'
        '(> 50% = majority agree wetter)',
        fontweight='bold')

    fig.suptitle(
        f'{region} \u2014 Anomaly & Model Agreement \u00b7 RCP 4.5 \u00b7 BC Ensemble',
        fontsize=12, fontweight='bold')
    fig.tight_layout(pad=2.0, w_pad=3.0)
    out = os.path.join(OUT_DIR, f'Fig6_{region}_ModelAgreement.png')
    fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show();  print(f'Saved: {out}')


## Step 12: Summary statistics CSV

In [ ]:
rows = []
for region, bc_data, ens in [
    ('Indonesia', indo_bc, ens_indo),
    ('Russia',    rus_bc,  ens_rus),
]:
    for mn, d in bc_data.items():
        bm = float(d['baseline_mean'].mean())
        fm = float(d['future_mean'].mean())
        am = float(d['anomaly'].mean())
        rows.append({'Region': region, 'Model': mn, 'Type': 'Individual (BC)',
                     'Baseline_mm_yr': round(bm,1), 'Future_mm_yr': round(fm,1),
                     'Anomaly_mm_yr': round(am,1),
                     'Change_%': round(am/bm*100, 2) if bm else np.nan})

    bm_e = float(ens['baseline'].mean()); fm_e = float(ens['future'].mean())
    am_e = float(ens['anomaly'].mean())
    rows.append({'Region': region, 'Model': 'Ensemble Mean', 'Type': 'Ensemble (BC)',
                 'Baseline_mm_yr': round(bm_e,1), 'Future_mm_yr': round(fm_e,1),
                 'Anomaly_mm_yr': round(am_e,1),
                 'Change_%': round(am_e/bm_e*100,2) if bm_e else np.nan})

df = pd.DataFrame(rows)
out_csv = os.path.join(OUT_DIR, 'Final_Summary_Statistics.csv')
df.to_csv(out_csv, index=False)
print('=' * 85)
print('FINAL BIAS-CORRECTED PRCPTOT SUMMARY')
print('=' * 85)
print(df.to_string(index=False))
print(f'\nSaved: {out_csv}')


## Step 13: CWSI distribution table

In [ ]:
# CWSI distribution table

cwsi_rows = []
for region, ens in [('Indonesia', ens_indo), ('Russia', ens_rus)]:
    lo, mo, hi = _stress_pcts(ens['stress'])
    cwsi_rows.append({
        'Region':               region,
        'Mean_WSI_%':           round(float(ens['wsi'].mean()), 2),
        'Min_WSI_%':            round(float(ens['wsi'].min()),  2),
        'Max_WSI_%':            round(float(ens['wsi'].max()),  2),
        'Low_stress_%area':     round(lo, 1),
        'Moderate_stress_%area':round(mo, 1),
        'High_stress_%area':    round(hi, 1),
        'Total_%':              round(lo+mo+hi, 1),
    })

cwsi_df = pd.DataFrame(cwsi_rows)
out_csv = os.path.join(OUT_DIR, 'Final_CWSI_Summary.csv')
cwsi_df.to_csv(out_csv, index=False)
print('=' * 75)
print('CWSI WATER STRESS DISTRIBUTION')
print('=' * 75)
print(cwsi_df.to_string(index=False))
print(f'\nSaved: {out_csv}')


## Step 14: Export final ensemble NetCDFs

In [ ]:
exports = {
    'Final_Ensemble_Baseline_Indonesia.nc':    ens_indo['baseline'],
    'Final_Ensemble_Future_Indonesia.nc':      ens_indo['future'],
    'Final_Ensemble_Anomaly_Indonesia.nc':     ens_indo['anomaly'],
    'Final_Ensemble_Uncertainty_Indonesia.nc': ens_indo['std'],
    'Final_Ensemble_WSI_Indonesia.nc':         ens_indo['wsi'],
    'Final_Ensemble_WaterStress_Indonesia.nc': ens_indo['stress'],
    'Final_Ensemble_Baseline_Russia.nc':       ens_rus['baseline'],
    'Final_Ensemble_Future_Russia.nc':         ens_rus['future'],
    'Final_Ensemble_Anomaly_Russia.nc':        ens_rus['anomaly'],
    'Final_Ensemble_Uncertainty_Russia.nc':    ens_rus['std'],
    'Final_Ensemble_WSI_Russia.nc':            ens_rus['wsi'],
    'Final_Ensemble_WaterStress_Russia.nc':    ens_rus['stress'],
}
print(f'Exporting {len(exports)} NetCDF files \u2192 {OUT_DIR}')
for fname, da in exports.items():
    fp = os.path.join(OUT_DIR, fname)
    da.to_netcdf(fp)
    print(f'  \u2713 {fname}  ({os.path.getsize(fp)/1024:.1f} KB)')
print('\nDone.')

## Step 15: Freshwater quality implication narrative

In [ ]:
# Freshwater implication narrative

print('=' * 70)
print('FRESHWATER QUALITY IMPLICATION SUMMARY')
print(f'RCP 4.5 | {FUTURE_LABEL} vs baseline | Bias-Corrected Ensemble')
print('=' * 70)

for region, ens in [('INDONESIA', ens_indo), ('RUSSIA', ens_rus)]:
    bm  = float(ens['baseline'].mean())
    fm  = float(ens['future'].mean())
    am  = float(ens['anomaly'].mean())
    wsi = float(ens['wsi'].mean())
    sd  = float(ens['std'].mean())
    lo, mo, hi = _stress_pcts(ens['stress'])
    pct_inc = float(ens['pct_increase'].mean())

    agree     = 'majority agree wetter' if pct_inc > 50 else 'majority agree drier'
    conf      = ('HIGH'     if pct_inc > 75 or pct_inc < 25 else
                 'MODERATE' if pct_inc > 60 or pct_inc < 40 else 'LOW')
    direction = 'wetter (+)' if am > 0 else 'drier (\u2212)'

    print(f'\n\u2500\u2500 {region} \u2500\u2500')
    print(f'  Baseline PRCPTOT        : {bm:.1f} mm/yr')
    print(f'  Future   PRCPTOT        : {fm:.1f} mm/yr')
    print(f'  Anomaly                 : {am:+.1f} mm/yr  ({wsi:+.1f}%)')
    print(f'  Ensemble uncertainty    : \u00b1{sd:.1f} mm/yr')
    print(f'  Dominant stress class   : Low {lo:.0f}% \u00b7 Mod {mo:.0f}% \u00b7 High {hi:.0f}%')
    print(f'  Model agreement         : {pct_inc:.0f}% models wetter \u2192 {agree}')
    print(f'  Confidence in direction : {conf}')
    print(f'\n  Freshwater implication:')
    if am > 0:
        print(f'    \u2192 Increased precipitation ({direction}) may enhance river discharge')
        print(f'       and groundwater recharge, reducing freshwater stress.')
        print(f'    \u2192 If concentrated in wet seasons, dry-season vulnerability may persist.')
        print(f'    \u2192 Higher runoff can increase sediment/nutrient loading.')
    else:
        print(f'    \u2192 Reduced precipitation ({direction}) signals decreased availability.')
        print(f'    \u2192 Reduced dilution raises pollutant concentrations in rivers.')
        print(f'    \u2192 Nutrient concentration risk rises as base flow diminishes.')
    print(f'\n    Uncertainty \u00b1{sd:.0f} mm/yr: treat as a range, not a single projection.')

print('\n' + '=' * 70)


## Step 16: Complete

In [ ]:
print('\n' + '=' * 60)
print('  FINAL ANALYSIS COMPLETE')
print('=' * 60)
print(f'\nAll outputs saved to: {OUT_DIR}')
print()
for f in sorted(os.listdir(OUT_DIR)):
    kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:<55} ({kb:.1f} KB)')
